<a href="https://colab.research.google.com/github/roro10212/ECEP/blob/master/content/es/docencia/python/ejercicios/soluciones/tipos-datos/ejercicio1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Ejercicio
Escribir un programa que muestre por pantalla la frase `¡Hola mundo!`

In [3]:
# Weinstein + screener en Google Colab (gratis) - versión 2
# Uso: pega TODO este código en una celda de Colab y ejecútala (botón de play).

import subprocess, sys
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
    import yfinance as yf

import pandas as pd

# ---------- Tu lista (edítala) ----------
# Chile en Yahoo termina en .SN. Verifica cada símbolo en finance.yahoo.com
TICKERS = [
    "VOO", "SPYG", "VGT",
    "VTV", "IWD", "XLF", "SCHD", "XLE", "IWM", "IJR", "IJH", "DIA", "XLI", "XLU", "XLV", "MDY", "RSP",
    "SQM-B.SN", "FALABELLA.SN", "COPEC.SN", "CHILE.SN",
]

# ---------- Etapas (iguales al indicador de TradingView) ----------
SMA_LEN   = 30     # semanas
SLOPE_LEN = 4      # semanas para medir pendiente
FLAT_THR  = 0.5    # % bajo el cual la SMA se considera plana
MAX_DIST  = 15.0   # % máx. de distancia a la SMA para "Etapa 2 temprana"
PERF_LEN  = 26     # semanas para el rendimiento (26 ≈ 6 meses)

# ---------- Filtros (los de tu screener) ----------
MIN_PERF        = 5.0              # Rend. 6M mayor que 5%
MAX_EXPENSE     = 0.3              # Ratio de gastos menor que 0,3 %
MIN_AUM         = 1_000_000        # AUM mayor que 1 M USD (súbelo a 1_000_000_000 para exigir fondos grandes)
MIN_DOLLAR_VOL  = 300_000_000      # Precio x volumen mayor que 300 M USD (volumen promedio de 3 meses)
DAILY_SMA       = 50               # Precio sobre SMA 50 diaria

STAGE_NAMES = {1: "1 · Base", 2: "2 · Avance", 3: "3 · Techo", 4: "4 · Declive"}
NAN = float("nan")


def classify(close: pd.Series):
    """Misma lógica del indicador de Pine, sobre cierres semanales."""
    ma = close.rolling(SMA_LEN).mean()
    slope = (ma / ma.shift(SLOPE_LEN) - 1) * 100
    stage = 1
    for c, m, s in zip(close, ma, slope):
        if pd.isna(m) or pd.isna(s):
            continue
        if s > FLAT_THR:
            if c > m:
                stage = 2
            else:
                stage = 2 if stage == 2 else 1
        elif s < -FLAT_THR:
            stage = 4 if c < m else 3
        else:
            stage = 3 if stage in (2, 3) else 1
    last_close, last_ma, last_slope = close.iloc[-1], ma.iloc[-1], slope.iloc[-1]
    dist = (last_close / last_ma - 1) * 100
    perf = (last_close / close.iloc[-1 - PERF_LEN] - 1) * 100 if len(close) > PERF_LEN else NAN
    return stage, last_slope, dist, perf


def get_info(ticker):
    """Ratio de gastos, AUM y volumen promedio desde Yahoo (pueden faltar)."""
    try:
        info = yf.Ticker(ticker).info or {}
    except Exception:
        info = {}
    exp = info.get("netExpenseRatio")             # suele venir en % (0,03 = 0,03 %)
    if exp is None and info.get("annualReportExpenseRatio") is not None:
        exp = info["annualReportExpenseRatio"] * 100   # viene como fracción
    return {"expense": exp, "aum": info.get("totalAssets"), "avgvol": info.get("averageVolume")}


def passes(value, test):
    """Si el dato falta, no descarta el símbolo (se avisa en 'Datos faltantes')."""
    if value is None or pd.isna(value):
        return True
    return bool(test(value))


def analyze(weekly: dict, daily: dict, infos: dict) -> pd.DataFrame:
    rows = []
    for t, wk in weekly.items():
        wk = wk.dropna()
        if len(wk) < SMA_LEN + SLOPE_LEN + 1:
            rows.append({"Símbolo": t, "Etapa": "sin datos suficientes"})
            continue
        stage, slope, dist, perf = classify(wk)

        dy = daily.get(t, pd.Series(dtype=float)).dropna()
        price = dy.iloc[-1] if len(dy) else wk.iloc[-1]
        sma_d = dy.rolling(DAILY_SMA).mean().iloc[-1] if len(dy) >= DAILY_SMA else NAN
        above_d = None if pd.isna(sma_d) else bool(price > sma_d)

        inf = infos.get(t, {})
        exp, aum, avgvol = inf.get("expense"), inf.get("aum"), inf.get("avgvol")
        dollar_vol = avgvol * price if avgvol else None

        missing = [n for n, v in (("gastos", exp), ("AUM", aum), ("volumen", dollar_vol), ("SMA diaria", above_d))
                   if v is None]

        cond = [
            passes(perf, lambda v: v >= MIN_PERF),
            dist > 0,                                   # precio sobre SMA 30 semanal
            above_d is not False,                       # precio sobre SMA 50 diaria
            passes(exp, lambda v: v <= MAX_EXPENSE),
            passes(aum, lambda v: v >= MIN_AUM),
            passes(dollar_vol, lambda v: v >= MIN_DOLLAR_VOL),
        ]

        rows.append({
            "Símbolo": t,
            "Etapa": STAGE_NAMES[stage],
            "Pend. SMA 4 sem %": round(slope, 2),
            "Dist. a SMA %": round(dist, 2),
            f"Rend. {PERF_LEN} sem %": round(perf, 2),
            "Ratio gastos %": round(exp, 3) if exp is not None else NAN,
            "AUM (miles de M USD)": round(aum / 1e9, 2) if aum else NAN,
            "Precio x vol (M USD)": round(dollar_vol / 1e6, 1) if dollar_vol else NAN,
            "Sobre SMA 50 diaria": "?" if above_d is None else ("SÍ" if above_d else "NO"),
            "Cumple filtros": "SÍ" if all(cond) else "NO",
            "Etapa 2 temprana": "SÍ" if (stage == 2 and dist <= MAX_DIST) else "NO",
            "Datos faltantes": ", ".join(missing),
        })
    return pd.DataFrame(rows)


def rank_candidates(df: pd.DataFrame) -> pd.DataFrame:
    perf_col = f"Rend. {PERF_LEN} sem %"
    if "Cumple filtros" not in df.columns:
        return pd.DataFrame()
    cand = df[df["Cumple filtros"] == "SÍ"].copy()
    if cand.empty:
        return cand
    cand["_early"] = (cand["Etapa 2 temprana"] == "SÍ").astype(int)
    cand = cand.sort_values(["_early", perf_col], ascending=[False, False]).drop(columns="_early")
    return cand.reset_index(drop=True)


def perf_style(v):
    if pd.isna(v):
        return ""
    if v >= 20:
        return "background-color:#006400;color:white"   # verde oscuro
    if v >= 10:
        return "background-color:#32CD32;color:black"   # verde lima
    if v >= 2:
        return "background-color:#DCF0DE;color:black"   # verde muy tenue
    return "background-color:#E0E0E0;color:black"


def stage_style(v):
    colors = {"1": "#9E9E9E", "2": "#4CAF50", "3": "#FF9800", "4": "#F44336"}
    k = str(v)[:1]
    return f"background-color:{colors[k]};color:white" if k in colors else ""


def show(df, title):
    print("\n" + title)
    if df.empty:
        print("(ninguno cumple)")
        return
    perf_col = f"Rend. {PERF_LEN} sem %"
    if perf_col not in df.columns:
        print(df.to_string(index=False))
        return
    styled = df.style.apply(lambda col: [perf_style(v) for v in col], subset=[perf_col]) \
                     .apply(lambda col: [stage_style(v) for v in col], subset=["Etapa"])
    try:
        from IPython.display import display
        display(styled)
    except Exception:
        print(df.to_string(index=False))


def main():
    wk_raw = yf.download(TICKERS, period="max", interval="1wk", auto_adjust=True, progress=False, group_by="ticker")
    dy_raw = yf.download(TICKERS, period="1y", interval="1d", auto_adjust=True, progress=False, group_by="ticker")
    weekly, daily, infos = {}, {}, {}
    for t in TICKERS:
        for raw, store in ((wk_raw, weekly), (dy_raw, daily)):
            try:
                store[t] = raw[t]["Close"]
            except Exception:
                store[t] = pd.Series(dtype=float)
        infos[t] = get_info(t)

    df = analyze(weekly, daily, infos)
    show(rank_candidates(df), "CANDIDATOS: cumplen tus filtros (Etapa 2 temprana primero, luego mayor rendimiento)")
    show(df, "TABLA COMPLETA")
    df.to_csv("weinstein_resultado.csv", index=False)
    print("\nGuardado en weinstein_resultado.csv (panel de archivos de Colab).")
    print("'?' = Yahoo no entregó el dato; en ese caso el filtro no descarta el símbolo.")
    print("La última barra es la semana en curso. Esto no es una recomendación de inversión.")


if __name__ == "__main__":
    main()


CANDIDATOS: cumplen tus filtros (Etapa 2 temprana primero, luego mayor rendimiento)


,Símbolo,Etapa,Pend. SMA 4 sem %,Dist. a SMA %,Rend. 26 sem %,Ratio gastos %,AUM (miles de M USD),Precio x vol (M USD),Sobre SMA 50 diaria,Cumple filtros,Etapa 2 temprana,Datos faltantes
0,VGT,2 · Avance,4.180000,14.880000,44.390000,0.090000,170.650000,517.400000,SÍ,SÍ,SÍ,
1,VOO,2 · Avance,1.660000,5.300000,17.980000,0.030000,1756.880000,5035.000000,SÍ,SÍ,SÍ,
2,CHILE.SN,2 · Avance,1.850000,4.880000,12.360000,nan,nan,9914.000000,SÍ,SÍ,SÍ,"gastos, AUM"
3,XLE,2 · Avance,2.100000,7.520000,7.420000,0.080000,41.440000,1894.400000,SÍ,SÍ,SÍ,
4,FALABELLA.SN,3 · Techo,0.380000,7.900000,8.380000,nan,nan,2096.500000,SÍ,SÍ,NO,"gastos, AUM"



TABLA COMPLETA


,Símbolo,Etapa,Pend. SMA 4 sem %,Dist. a SMA %,Rend. 26 sem %,Ratio gastos %,AUM (miles de M USD),Precio x vol (M USD),Sobre SMA 50 diaria,Cumple filtros,Etapa 2 temprana,Datos faltantes
0,VOO,2 · Avance,1.660000,5.300000,17.980000,0.030000,1756.880000,5035.000000,SÍ,SÍ,SÍ,
1,SPYG,2 · Avance,2.320000,8.130000,25.860000,0.040000,54.460000,276.600000,SÍ,NO,SÍ,
2,VGT,2 · Avance,4.180000,14.880000,44.390000,0.090000,170.650000,517.400000,SÍ,SÍ,SÍ,
3,VTV,2 · Avance,1.150000,2.430000,11.330000,0.030000,262.300000,589.700000,NO,NO,SÍ,
4,IWD,2 · Avance,1.770000,4.370000,16.500000,0.180000,83.950000,407.700000,NO,NO,SÍ,
5,XLF,2 · Avance,1.100000,0.060000,8.760000,0.080000,54.590000,1761.200000,NO,NO,SÍ,
6,SCHD,2 · Avance,1.060000,1.760000,8.780000,0.060000,112.340000,740.100000,NO,NO,SÍ,
7,XLE,2 · Avance,2.100000,7.520000,7.420000,0.080000,41.440000,1894.400000,SÍ,SÍ,SÍ,
8,IWM,2 · Avance,1.230000,-0.160000,12.590000,0.190000,80.460000,5871.800000,NO,NO,SÍ,
9,IJR,2 · Avance,1.080000,0.040000,11.170000,0.060000,109.250000,419.700000,NO,NO,SÍ,



Guardado en weinstein_resultado.csv (panel de archivos de Colab).
'?' = Yahoo no entregó el dato; en ese caso el filtro no descarta el símbolo.
La última barra es la semana en curso. Esto no es una recomendación de inversión.


## Solución

In [4]:
print("¡Hola Mundo!")

¡Hola Mundo!
